# Summit.OS — ReID Embedder Training

> **First:** Runtime → Change runtime type → T4 GPU → Save

Trains on **Market-1501** — 12,936 real camera crops, 751 identities.

Outputs → drop into `packages/ml/models/`

Expected: ~35 min on T4.

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
print(r.stdout.strip() if r.returncode==0 else 'NO GPU — change runtime type first!')

In [ ]:
!pip install -q gdown onnx onnxruntime onnxscript
print('Done.')

In [ ]:
import os, zipfile
TRAIN_DIR='/content/data/Market-1501-v15.09.15/bounding_box_train'
EXT_DIR='/content/data/Market-1501-v15.09.15'
if not os.path.isdir(TRAIN_DIR):
    print('Downloading Market-1501 (~1.6 GB)...')
    !gdown 0B8-rUzbwVRk0c054eEozWG9COHM -O /content/market1501.zip
    with zipfile.ZipFile('/content/market1501.zip') as z: z.extractall('/content/data')
    os.remove('/content/market1501.zip')
else:
    print('Already downloaded.')
from collections import Counter
imgs=[f for f in os.listdir(TRAIN_DIR) if f.endswith('.jpg')]
counts=Counter(f[:4] for f in imgs if f[:4] not in('-1','0000'))
print(f'{len(imgs)} images, {len(counts)} identities, {sum(counts.values())/len(counts):.1f} crops/id')

In [ ]:
import torch,torch.nn as nn,torch.nn.functional as F,numpy as np,random,json,warnings,os
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from collections import defaultdict

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)

AUG=transforms.Compose([
    transforms.Resize((128,64)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.3,contrast=0.3,saturation=0.3,hue=0.05),
    transforms.RandomAffine(degrees=0,translate=(0.05,0.1)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3,scale=(0.02,0.15)),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
INFER=transforms.Compose([
    transforms.Resize((128,64)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class Market1501Dataset(Dataset):
    def __init__(self,train_dir,n=10000):
        self.id2p=defaultdict(list)
        for f in sorted(os.listdir(train_dir)):
            if not f.endswith('.jpg'): continue
            pid=f[:4]
            if pid in('-1','0000'): continue
            self.id2p[pid].append(os.path.join(train_dir,f))
        self.ids=[p for p,v in self.id2p.items() if len(v)>=2]
        self.n=n
        print(f'  {len(self.ids)} identities, {sum(len(v) for v in self.id2p.values())} images')
    def __len__(self): return self.n
    def _load(self,p): return AUG(Image.open(p).convert('RGB'))
    def __getitem__(self,idx):
        rng=random.Random(idx*7919)
        aid=rng.choice(self.ids)
        ni=rng.randint(0,len(self.ids)-2)
        nid=self.ids[ni if self.ids[ni]!=aid else ni+1]
        ap,pp=rng.sample(self.id2p[aid],2)
        return self._load(ap),self._load(pp),self._load(rng.choice(self.id2p[nid]))

class IR(nn.Module):
    def __init__(self,ic,oc,s=1,e=6):
        super().__init__()
        m=ic*e; self.r=s==1 and ic==oc
        L=[]
        if e!=1: L+=[nn.Conv2d(ic,m,1,bias=False),nn.BatchNorm2d(m),nn.ReLU6(True)]
        L+=[nn.Conv2d(m,m,3,stride=s,padding=1,groups=m,bias=False),nn.BatchNorm2d(m),nn.ReLU6(True),nn.Conv2d(m,oc,1,bias=False),nn.BatchNorm2d(oc)]
        self.c=nn.Sequential(*L)
    def forward(self,x): return self.c(x)+x if self.r else self.c(x)

class ReIDEmbedder(nn.Module):
    def __init__(self,d=128):
        super().__init__()
        self.s=nn.Sequential(nn.Conv2d(3,32,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(32),nn.ReLU6(True))
        self.b=nn.Sequential(IR(32,64,2),IR(64,64,1),IR(64,128,2),IR(128,128,1))
        self.p=nn.AdaptiveAvgPool2d(1); self.h=nn.Linear(128,d)
    def forward(self,x): return F.normalize(self.h(self.b(self.s(x)).mean([-2,-1])),p=2,dim=1)

print('Ready.')

In [ ]:
EPOCHS,BATCH,TRIPLETS=40,128,10000
OUTDIR=Path('/content/models'); OUTDIR.mkdir(exist_ok=True)

ds=Market1501Dataset(TRAIN_DIR,TRIPLETS)
loader=DataLoader(ds,batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)
model=ReIDEmbedder().to(device)
opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
criterion=nn.TripletMarginLoss(margin=0.3,p=2)

print(f'Params: {sum(p.numel() for p in model.parameters()):,}  {EPOCHS} epochs  {TRIPLETS} triplets/epoch')
for epoch in range(1,EPOCHS+1):
    model.train(); losses=[]
    for a,p,n in loader:
        a,p,n=a.to(device),p.to(device),n.to(device)
        l=criterion(model(a),model(p),model(n))
        opt.zero_grad(); l.backward(); opt.step(); losses.append(l.item())
    sch.step()
    if epoch%10==0 or epoch==1 or epoch==EPOCHS:
        print(f'Epoch {epoch:>3}/{EPOCHS}  loss={np.mean(losses):.4f}  lr={sch.get_last_lr()[0]:.6f}')
print('Done.')

In [ ]:
# Recall@1 on real query set
def embed(path):
    model.eval(); embs,ids=[],[]
    with torch.no_grad():
        batch,bids=[],[]
        for f in sorted(os.listdir(path)):
            if not f.endswith('.jpg'): continue
            pid=f[:4]
            if pid in('-1','0000'): continue
            batch.append(INFER(Image.open(os.path.join(path,f)).convert('RGB'))); bids.append(pid)
            if len(batch)==256:
                embs.append(model(torch.stack(batch).to(device)).cpu().numpy()); ids+=bids; batch,bids=[],[]
        if batch: embs.append(model(torch.stack(batch).to(device)).cpu().numpy()); ids+=bids
    return np.vstack(embs),ids

q_e,q_i=embed(f'{EXT_DIR}/query')
g_e,g_i=embed(f'{EXT_DIR}/bounding_box_test')
preds=np.argmax(q_e@g_e.T,axis=1)
recall1=sum(g_i[p]==q for p,q in zip(preds,q_i))/len(q_i)
print(f'Recall@1: {recall1:.3f}  (good lightweight models: 0.75-0.85)')

In [ ]:
onnx_path=OUTDIR/'reid_embedder.onnx'
cfg_path=OUTDIR/'reid_embedder_config.json'
model.eval()
dummy=torch.randn(1,3,128,64,device=device)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    try:
        torch.onnx.export(model,dummy,str(onnx_path),input_names=['image'],output_names=['embedding'],dynamic_axes={'image':{0:'batch'},'embedding':{0:'batch'}},opset_version=17)
    except Exception:
        torch.onnx.export(model.cpu(),dummy.cpu(),str(onnx_path),input_names=['image'],output_names=['embedding'],dynamic_axes={'image':{0:'batch'},'embedding':{0:'batch'}},opset_version=12)
with open(cfg_path,'w') as f:
    json.dump({'input_size':[128,64],'embedding_dim':128,'channels':3,'normalize':True,'backend':'pytorch','trained_on':'Market-1501 (12936 real crops, 751 identities)','recall_at_1':round(float(recall1),4)},f,indent=2)
print(f'ONNX: {onnx_path}  ({onnx_path.stat().st_size/1024:.1f} KB)')
print(f'Config: {cfg_path}')

In [ ]:
from google.colab import files
files.download(str(onnx_path))
files.download(str(cfg_path))